In [1]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd

In [8]:
# Define allowed categories
allowed_categories = {"cs.AI", "cs.CL", "stat.ML", "math.OC", "cs.LG"}

# Build the search query: articles that have any of the specified categories
query = " OR ".join([f"cat:{cat}" for cat in allowed_categories])

# API parameters
base_url = "http://export.arxiv.org/api/query"
params = {
    "search_query": query,
    "start": 0,
    "max_results": 200   # You can adjust this as needed
}

In [25]:
response = requests.get(base_url, params=params)
response.raise_for_status()

root = ET.fromstring(response.content)

In [26]:
dataset = []

for entry in root.findall("{http://www.w3.org/2005/Atom}entry"):
    title = entry.find("{http://www.w3.org/2005/Atom}title").text.strip()
    summary = entry.find("{http://www.w3.org/2005/Atom}summary").text.strip()
    link = entry.find("{http://www.w3.org/2005/Atom}id").text.strip()
    
    # Extract categories (primary and secondary)
    primary_cat = entry.find("{http://arxiv.org/schemas/atom}primary_category").attrib["term"]
    all_cats = {primary_cat}
    for cat in entry.findall("{http://www.w3.org/2005/Atom}category"):
        all_cats.add(cat.attrib["term"])

    # Filter the categories to only keep those in allowed_categories
    filtered_labels = list(all_cats.intersection(allowed_categories))
    
    # Only include the article if it has at least one of the allowed categories
    if filtered_labels:
        dataset.append({
            "article title": title,
            "article abstract": summary,
            "link": link,
            "labels": filtered_labels
        })


In [27]:
# Convert to a DataFrame (optional)
df = pd.DataFrame(dataset)

# Print out some sample rows
df.head()

,article title,article abstract,link,labels
0,Epidemic control via stochastic optimal control,We study the problem of optimal control of the...,http://arxiv.org/abs/2004.06680v3,"[q-fin.CP, math.OC, econ.EM]"
1,Portfolio risk allocation through Shapley value,We argue that using the Shapley value of coope...,http://arxiv.org/abs/2103.05453v1,"[q-fin.CP, math.OC, econ.EM]"
2,Financial Time Series Forecasting using CNN an...,Time series forecasting is important across va...,http://arxiv.org/abs/2304.04912v1,"[q-fin.CP, econ.EM, cs.AI]"
3,Data Scaling Effect of Deep Learning in Financ...,"For years, researchers investigated the applic...",http://arxiv.org/abs/2309.02072v5,"[q-fin.CP, econ.EM, cs.AI]"
4,Optimal Text-Based Time-Series Indices,We propose an approach to construct text-based...,http://arxiv.org/abs/2405.10449v1,"[q-fin.CP, econ.EM, cs.AI]"


In [28]:
df['labels'].value_counts()

labels
[econ.EM, stat.ML]              36
[q-fin.CP, math.OC]             28
[econ.EM, cs.AI, stat.ML]       27
[q-fin.CP, econ.EM]             22
[q-fin.CP, cs.AI]               18
[math.OC, econ.EM, stat.ML]     16
[math.OC, econ.EM]              14
[q-fin.CP, math.OC, stat.ML]    12
[q-fin.CP, stat.ML]             12
[q-fin.CP, cs.AI, stat.ML]       5
[q-fin.CP, econ.EM, cs.AI]       4
[q-fin.CP, econ.EM, stat.ML]     3
[q-fin.CP, math.OC, econ.EM]     2
[math.OC, econ.EM, cs.AI]        1
Name: count, dtype: int64

In [29]:
# count incidence of each label
label_counts = df['labels'].explode().value_counts()
print(label_counts)

labels
econ.EM     125
stat.ML     111
q-fin.CP    106
math.OC      73
cs.AI        55
Name: count, dtype: int64


In [9]:
from datasets import load_dataset
import numpy as np

ds = load_dataset("TimSchopf/arxiv_categories", "default")

df_train = ds['train'].to_pandas()
df_test = ds['test'].to_pandas() 
df_val = ds['validation'].to_pandas()

# convert column 'categories' from ndarray to list
df_train['categories'] = df_train['categories'].apply(lambda x: x.tolist())
df_test['categories'] = df_test['categories'].apply(lambda x: x.tolist())
df_val['categories'] = df_val['categories'].apply(lambda x: x.tolist())

In [10]:
def clean_element(lst):
    final = []
    for elem in lst:
        clean = elem.split('->')[-1]
        final.append(clean)
    return final

df_test['categories'] = df_test['categories'].apply(clean_element)

In [11]:
label_list = df_test['categories'].explode().unique()

# for all labels in label list, check, in the test set, which labels appear together the most in pairs
# for each pair, count how many times they appear together

from itertools import combinations
from collections import Counter

def count_pairs(df, label_list):
    pairs = list(combinations(label_list, 2))
    pair_counts = Counter()
    for labels in df['categories']:
        for pair in pairs:
            if pair[0] in labels and pair[1] in labels:
                pair_counts[pair] += 1
    return pair_counts

pair_counts = count_pairs(df_test, label_list)
pair_counts

Counter({('gr-qc', 'hep-th'): 445,
         ('math.MP', 'math-ph'): 419,
         ('cs.LG', 'stat.ML'): 332,
         ('cs.IT', 'math.IT'): 322,
         ('hep-th', 'hep-ph'): 272,
         ('hep-ph', 'nucl-th'): 217,
         ('cs.LG', 'cs.AI'): 215,
         ('cs.CV', 'cs.LG'): 207,
         ('hep-ex', 'hep-ph'): 204,
         ('gr-qc', 'astro-ph.CO'): 156,
         ('math.ST', 'stat.TH'): 138,
         ('cond-mat.mes-hall', 'cond-mat.mtrl-sci'): 130,
         ('cs.CV', 'eess.IV'): 128,
         ('cs.SY', 'eess.SY'): 125,
         ('cond-mat.str-el', 'cond-mat.supr-con'): 124,
         ('hep-ph', 'astro-ph.CO'): 113,
         ('math.NA', 'cs.NA'): 110,
         ('gr-qc', 'hep-ph'): 109,
         ('hep-th', 'astro-ph.CO'): 106,
         ('astro-ph.GA', 'astro-ph.SR'): 99,
         ('cs.AI', 'cs.CL'): 99,
         ('astro-ph.GA', 'astro-ph.CO'): 98,
         ('nucl-th', 'nucl-ex'): 95,
         ('hep-ph', 'hep-lat'): 93,
         ('astro-ph', 'gr-qc'): 92,
         ('cond-mat.str-el', 

In [12]:
# drop all categories that are not in allowed_categories, from df_test
def filter_categories(lst):
    return [x for x in lst if x in allowed_categories]

df_test['categories'] = df_test['categories'].apply(filter_categories)

df_test['categories'].value_counts()

categories
[]                           18607
[cs.LG]                        413
[cs.CL]                        294
[cs.LG, stat.ML]               261
[math.OC]                      242
[cs.AI]                        184
[cs.AI, cs.LG]                 143
[cs.AI, cs.CL]                  68
[cs.CL, cs.LG]                  49
[cs.AI, cs.LG, stat.ML]         41
[cs.AI, cs.CL, cs.LG]           31
[cs.LG, math.OC, stat.ML]       20
[stat.ML]                       17
[cs.LG, math.OC]                17
[cs.CL, cs.LG, stat.ML]         10
Name: count, dtype: int64